In [1]:
import pandas as pd
import sys
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_classification_report, flat_f1_score
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

!{sys.executable} -m pip install sklearn-crfsuite
dataset_path = '../data/NER dataset.csv'
test_path = '../test_sets/NER-test.tsv'


def read_table_with_fallback(filepath, **kwargs):
    """Read a delimited text file using a small set of common encodings."""
    encodings = kwargs.pop('encodings', ['utf-8', 'utf-8-sig', 'cp1252', 'latin1'])
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(filepath, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def parse_ner_csv(filepath):
    """
    Parse NER dataset CSV format.
    Returns list of sentences, where each sentence is a list of token dictionaries.
    """
    df = read_table_with_fallback(filepath)
    sentences = []
    current_sentence = []
    current_sent_id = None

    for idx, row in df.iterrows():
        sent_id = str(row['Sentence #']).split(':')[0].strip()

        #Check if on a new sentence
        if sent_id != current_sent_id:
            if current_sentence:
                sentences.append(current_sentence)
            current_sentence = []
            current_sent_id = sent_id

        current_sentence.append({
            'word': row['Word'],
            'pos': row['POS'],
            'ner': row['Tag']
        })

    #last sentence
    if current_sentence:
        sentences.append(current_sentence)

    return sentences


def parse_ner_tsv(filepath):
    """
    Parse NER-test.tsv format.
    Returns list of sentences, where each sentence is a list of token dictionaries.
    """
    df = read_table_with_fallback(filepath, sep='\t')
    sentences = []
    current_sentence = []
    current_sent_id = None

    for idx, row in df.iterrows():
        sent_id = row['sentence id']

        #Check if on a new sentence
        if sent_id != current_sent_id:
            if current_sentence:
                sentences.append(current_sentence)
            current_sentence = []
            current_sent_id = sent_id

        current_sentence.append({
            'word': row['token'],
            'pos': '',
            'ner': row['BIO NER tag']
        })

    if current_sentence:
        sentences.append(current_sentence)

    return sentences


all_kaggle_sentences = parse_ner_csv(dataset_path)
docent_test_sentences = parse_ner_tsv(test_path)
train_sentences, internal_test_sentences = train_test_split(all_kaggle_sentences, test_size=0.2, random_state=42)

print(f"Kaggle zinnen voor Training: {len(train_sentences)}")
print(f"Kaggle zinnen voor Interne Test: {len(internal_test_sentences)}")
print(f"Docent zinnen voor Eindevaluatie: {len(docent_test_sentences)}")

# %%
#Normalize training/test labels to common CoNLL-style space
#Keep BIO prefix, map entity type part.
ENTITY_MAP = {
    "geo": "LOC",
    "gpe": "LOC",
    "loc": "LOC",
    "org": "ORG",
    "per": "PER",
    "art": "MISC",
    "eve": "MISC",
    "nat": "MISC",
    "tim": "MISC",
    "misc": "MISC",
    "LOC": "LOC",
    "ORG": "ORG",
    "PER": "PER",
    "MISC": "MISC",
}


def normalize_tag(tag):
    if not isinstance(tag, str):
        return "O"
    tag = tag.strip()
    if tag == "O" or tag == "":
        return "O"
    if "-" not in tag:
        return "O"
    bio, ent = tag.split("-", 1)
    ent_norm = ENTITY_MAP.get(ent, ENTITY_MAP.get(ent.lower(), "MISC"))
    bio = "B" if bio.upper().startswith("B") else "I"
    return f"{bio}-{ent_norm}"


def normalize_sentences_labels(sentences):
    for sent in sentences:
        for tok in sent:
            tok["ner"] = normalize_tag(tok.get("ner", "O"))
    return sentences


train_sentences = normalize_sentences_labels(train_sentences)
internal_test_sentences = normalize_sentences_labels(internal_test_sentences)
docent_test_sentences = normalize_sentences_labels(docent_test_sentences)

#check voor alle drie de sets
train_label_set = sorted({tok["ner"] for s in train_sentences for tok in s})
internal_label_set = sorted({tok["ner"] for s in internal_test_sentences for tok in s})
test_label_set = sorted({tok["ner"] for s in docent_test_sentences for tok in s})

print("Train labels:", train_label_set)
print("Internal Test labels:", internal_label_set)
print("Docent Test labels:", test_label_set)


# %%
def extract_ner_tags(sentences):
    """Extract all NER tags from sentences."""
    tags = []
    for sentence in sentences:
        for token in sentence:
            tags.append(token['ner'])
    return tags


def extract_ner_entities(sentences):
    """
    Extract entities (not just tags).
    Returns list of (entity_type, entity_text) tuples.
    """
    entities = []
    for sentence in sentences:
        current_type = None
        current_words = []

        for token in sentence:
            tag = token['ner']
            word = token['word']

            if tag == 'O':
                if current_words:
                    entity_text = ' '.join(current_words)
                    entities.append((current_type, entity_text))
                    current_words = []
                    current_type = None
            else:
                if '-' in tag:
                    bio, ent_type = tag.split('-', 1)
                else:
                    bio, ent_type = tag, tag

                if bio == 'B' or (current_type != ent_type and current_words):
                    if current_words:
                        entity_text = ' '.join(current_words)
                        entities.append((current_type, entity_text))
                    current_words = [word]
                    current_type = ent_type
                else:
                    current_words.append(word)
                    current_type = ent_type

        if current_words:
            entity_text = ' '.join(current_words)
            entities.append((current_type, entity_text))

    return entities


def get_entity_types(entities):
    """Extract entity types from entities list."""
    return [ent_type for ent_type, _ in entities]


def count_instances(sentences, tags, entities):
    """Count various instances in the dataset."""
    return {
        'num_sentences': len(sentences),
        'num_tokens': len(tags),
        'num_entities': len(entities),
        'unique_entity_types': len(set(get_entity_types(entities))),
        'unique_tags': len(set(tags))
    }


def sent2features(sent):
    features = []
    for i, token in enumerate(sent):
        word = '' if pd.isna(token.get('word', '')) else str(token.get('word', ''))
        pos = '' if pd.isna(token.get('pos', '')) else str(token.get('pos', ''))
        feat = {
            'bias': 1.0,
            'word.lower()': word.lower(),
            'word[-3:]': word[-3:],
            'word[-2:]': word[-2:],
            'word.isupper()': word.isupper(),
            'word.istitle()': word.istitle(),
            'word.isdigit()': word.isdigit(),
            'postag': pos,
            'postag[:2]': pos[:2] if pos else '',
        }

        if i > 0:
            prev_word = '' if pd.isna(sent[i - 1].get('word', '')) else str(sent[i - 1].get('word', ''))
            prev_pos = '' if pd.isna(sent[i - 1].get('pos', '')) else str(sent[i - 1].get('pos', ''))
            feat.update({
                '-1:word.lower()': prev_word.lower(),
                '-1:word.istitle()': prev_word.istitle(),
                '-1:word.isupper()': prev_word.isupper(),
                '-1:postag': prev_pos,
                '-1:postag[:2]': prev_pos[:2] if prev_pos else '',
            })
        else:
            feat['BOS'] = True

        if i < len(sent) - 1:
            next_word = '' if pd.isna(sent[i + 1].get('word', '')) else str(sent[i + 1].get('word', ''))
            next_pos = '' if pd.isna(sent[i + 1].get('pos', '')) else str(sent[i + 1].get('pos', ''))
            feat.update({
                '+1:word.lower()': next_word.lower(),
                '+1:word.istitle()': next_word.istitle(),
                '+1:word.isupper()': next_word.isupper(),
                '+1:postag': next_pos,
                '+1:postag[:2]': next_pos[:2] if next_pos else '',
            })
        else:
            feat['EOS'] = True

        features.append(feat)
    return features


def sent2labels(sent):
    return [tok["ner"] for tok in sent]


# =====================================================================
# 1. EXTRACT DATA FOR ANALYSIS (Gecorrigeerd naar de nieuwe sets)
# =====================================================================
train_tags = extract_ner_tags(train_sentences)
test_tags = extract_ner_tags(internal_test_sentences)

train_entities = extract_ner_entities(train_sentences)
test_entities = extract_ner_entities(internal_test_sentences)

# =====================================================================
# 2. DATA DISTRIBUTIONS & COUNTS
# =====================================================================
train_counts = count_instances(train_sentences, train_tags, train_entities)
test_counts = count_instances(internal_test_sentences, test_tags, test_entities)

counts_df = pd.DataFrame({
    'Train': train_counts,
    'Internal Test': test_counts
})
print("\n--- DATASET INSTANCES ---")
print(counts_df)

# Tag frequency distributions
train_tag_freq = Counter(train_tags)
test_tag_freq = Counter(test_tags)

all_tags = set(train_tag_freq.keys()) | set(test_tag_freq.keys())
tag_comparison = pd.DataFrame({
    'Train': [train_tag_freq[tag] for tag in sorted(all_tags)],
    'Internal Test': [test_tag_freq[tag] for tag in sorted(all_tags)]
}, index=sorted(all_tags))

tag_percentages = tag_comparison.div(tag_comparison.sum(axis=0), axis=1) * 100

print("\nNER TAG DISTRIBUTION (Counts)")
print(tag_comparison, '\n')
print("NER TAG DISTRIBUTION (Percentages)")
print(tag_percentages.round(2))

# Entity type frequencies
train_entity_types = [ent_type for ent_type, _ in train_entities]
test_entity_types = [ent_type for ent_type, _ in test_entities]

train_ent_freq = Counter(train_entity_types)
test_ent_freq = Counter(test_entity_types)

all_ent_types = set(train_ent_freq.keys()) | set(test_ent_freq.keys())
entity_comparison = pd.DataFrame({
    'Train': [train_ent_freq[ent_type] for ent_type in sorted(all_ent_types)],
    'Internal Test': [test_ent_freq[ent_type] for ent_type in sorted(all_ent_types)]
}, index=sorted(all_ent_types))

entity_percentages = entity_comparison.div(entity_comparison.sum(axis=0), axis=1) * 100

print("\nENTITY TYPE DISTRIBUTION (Counts)")
print(entity_comparison, '\n')
print("ENTITY TYPE DISTRIBUTION (Percentages)")
print(entity_percentages.round(2))

# =====================================================================
# 3. FEATURE EXTRACTION (Opgesplitst voor beide test-perspectieven)
# =====================================================================
X_train = [sent2features(s) for s in train_sentences]
y_train = [sent2labels(s) for s in train_sentences]

# System Perspective (Interne testset van ~19.000 zinnen)
X_test_internal = [sent2features(s) for s in internal_test_sentences]
y_test_internal = [sent2labels(s) for s in internal_test_sentences]

# Application Perspective (De 10 blinde testzinnen van de docent)
X_test_docent = [sent2features(s) for s in docent_test_sentences]
y_test_docent = [sent2labels(s) for s in docent_test_sentences]

# =====================================================================
# 4. MODEL TRAINING
# =====================================================================
crf = CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("\nTraining CRF model...")
crf.fit(X_train, y_train)

# =====================================================================
# 5. EVALUATIE PERSPECTIEF 1: System Perspective (Internal 20% Split)
# =====================================================================
y_pred_internal = crf.predict(X_test_internal)

flat_true_internal = [label for sent in y_test_internal for label in sent]
flat_pred_internal = [label for sent in y_pred_internal for label in sent]

print("\n=======================================================")
print("EVALUATION 1: SYSTEM PERSPECTIVE (Internal 80/20 Test Split)")
print("=======================================================")
print(f"Token accuracy: {accuracy_score(flat_true_internal, flat_pred_internal):.4f}")
print(f"Flat F1 (micro): {flat_f1_score(y_test_internal, y_pred_internal, average='micro'):.4f}")
print(f"Flat F1 (macro): {flat_f1_score(y_test_internal, y_pred_internal, average='macro'):.4f}")
print("\nClassification report (Internal Test):")
print(flat_classification_report(y_test_internal, y_pred_internal))

# =====================================================================
# 6. EVALUATIE PERSPECTIEF 2: Application Perspective (Docent 10 Sentences)
# =====================================================================
y_pred_docent = crf.predict(X_test_docent)

flat_true_docent = [label for sent in y_test_docent for label in sent]
flat_pred_docent = [label for sent in y_pred_docent for label in sent]

print("\n=======================================================")
print("EVALUATION 2: APPLICATION PERSPECTIVE (Professor's 10 Sentences)")
print("=======================================================")
print(f"Token accuracy: {accuracy_score(flat_true_docent, flat_pred_docent):.4f}")
print(f"Flat F1 (micro): {flat_f1_score(y_test_docent, y_pred_docent, average='micro'):.4f}")
print(f"Flat F1 (macro): {flat_f1_score(y_test_docent, y_pred_docent, average='macro'):.4f}")
print("\nClassification report (Professor's Test):")
print(flat_classification_report(y_test_docent, y_pred_docent, zero_division=0))



Kaggle zinnen voor Training: 76731
Kaggle zinnen voor Interne Test: 19183
Docent zinnen voor Eindevaluatie: 10
Train labels: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Internal Test labels: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Docent Test labels: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']

--- DATASET INSTANCES ---
                      Train  Internal Test
num_sentences         76731          19183
num_tokens           838775         209800
num_entities          92794          23135
unique_entity_types       4              4
unique_tags               9              9

NER TAG DISTRIBUTION (Counts)
         Train  Internal Test
B-LOC    42743          10771
B-MISC   17027           4217
B-ORG    16145           3998
B-PER    13662           3328
I-LOC     6055           1557
I-MISC    5701           1428
I-ORG    13511           3273
I-PER    13946           3305
O      